# Diagnostic accuracy metrics: sensitivity, specificity, PPV, ROC and calibration: worked solution

Worked solution for the exercise in the lesson **[Diagnostic accuracy metrics: sensitivity, specificity, PPV, ROC and calibration](https://ramkumarmd.com/medical-ai/foundations/diagnostic-accuracy-metrics)** (*Foundations* track, Medical AI course by Dr. Ram Kumar G). Try the [exercise notebook](06-diagnostic-accuracy-metrics.ipynb) first. Other answers can be just as good: the written parts show one reasonable answer, not the only one.

**In short:** Sensitivity and specificity describe how a test or model behaves in people with and without a condition. Predictive values tell you what a result means for the person in front of you, and they change with how common the condition is. AUROC measures how well a model ranks patients, and calibration checks whether its predicted probabilities can be taken at face value. A clinical AI model needs all of these, not one headline number.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Why PPV depends on prevalence

PPV for a test with 90% sensitivity and 80% specificity at different prevalences.

In [ ]:
def ppv(sensitivity, specificity, prevalence):
    true_pos = sensitivity * prevalence
    false_pos = (1 - specificity) * (1 - prevalence)
    return true_pos / (true_pos + false_pos)

for prevalence in [0.01, 0.05, 0.10, 0.30]:
    print(f"prevalence {prevalence:>4.0%}  PPV {ppv(0.90, 0.80, prevalence):.1%}")

# prevalence   1%  PPV 4.3%
# prevalence   5%  PPV 19.1%
# prevalence  10%  PPV 33.3%
# prevalence  30%  PPV 65.9%

### ROC curves and AUROC

Simulate a model's scores for 2,000 patients at 10% prevalence, then compute the metrics at a 0.5 threshold.

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, roc_auc_score

rng = np.random.default_rng(42)
n = 2000
y_true = rng.binomial(1, 0.10, size=n)  # 10% prevalence

# Patients with the condition get higher scores on average
logit = rng.normal(loc=np.where(y_true == 1, 1.5, -1.0), scale=1.0)
y_prob = 1 / (1 + np.exp(-logit))

y_pred = (y_prob >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

print(f"Sensitivity {tp / (tp + fn):.2f}   Specificity {tn / (tn + fp):.2f}")
print(f"PPV         {tp / (tp + fp):.2f}   NPV         {tn / (tn + fn):.2f}")
print(f"AUROC       {roc_auc_score(y_true, y_prob):.3f}")

# Sensitivity 0.92   Specificity 0.85
# PPV         0.41   NPV         0.99
# AUROC       0.958

### Calibration: can the probabilities be trusted?

Continuing from the previous block: compare predicted and observed risk in ten groups, and compute the Brier score.

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

observed, predicted = calibration_curve(y_true, y_prob, n_bins=10, strategy="quantile")
for p, o in zip(predicted, observed):
    print(f"predicted {p:.2f}   observed {o:.2f}")

print(f"Brier score {brier_score_loss(y_true, y_prob):.3f}")  # lower is better

# predicted 0.07   observed 0.00
# predicted 0.13   observed 0.00
# ...
# predicted 0.49   observed 0.04
# predicted 0.62   observed 0.20
# predicted 0.85   observed 0.76
# Brier score 0.120

## Exercise

**Goal:** Explore how threshold and prevalence change the metrics of a simulated risk model, and check its calibration.

**Data:** Synthetic patients generated in the notebook (no real patient data) (Generated in the notebook)

### Step 1

Run the simulation above and record sensitivity, specificity, PPV and NPV at thresholds 0.3, 0.5 and 0.7.

In [ ]:
import pandas as pd

def simulate(prevalence, n=2000, seed=42):
    """The worked example's simulation as a function: same model, any prevalence."""
    rng = np.random.default_rng(seed)
    y = rng.binomial(1, prevalence, size=n)
    logit = rng.normal(loc=np.where(y == 1, 1.5, -1.0), scale=1.0)
    return y, 1 / (1 + np.exp(-logit))

def metrics_at(y, prob, threshold):
    tn, fp, fn, tp = confusion_matrix(y, (prob >= threshold).astype(int)).ravel()
    return {"threshold": threshold, "sensitivity": tp / (tp + fn), "specificity": tn / (tn + fp),
            "PPV": tp / (tp + fp), "NPV": tn / (tn + fn)}

THRESHOLDS = [0.3, 0.5, 0.7]
y_true, y_prob = simulate(0.10)
print(pd.DataFrame([metrics_at(y_true, y_prob, t) for t in THRESHOLDS]).round(2).to_string(index=False))

Raising the threshold trades sensitivity for specificity: fewer patients are flagged, so fewer cases are
found but a larger share of the flagged patients truly have the condition (PPV rises).

### Step 2

Re-run the simulation at 2% and 30% prevalence without changing the model. Which metrics move, and which stay roughly the same?

In [ ]:
rows = []
for prevalence in [0.02, 0.10, 0.30]:
    y, prob = simulate(prevalence, n=20000)   # a larger sample keeps the 2% estimates stable
    m = metrics_at(y, prob, 0.5)
    rows.append({"prevalence": prevalence, **m, "AUROC": roc_auc_score(y, prob)})
print(pd.DataFrame(rows).drop(columns="threshold").round(2).to_string(index=False))

Sensitivity, specificity and AUROC stay roughly the same, because they are calculated within the
people with, or without, the condition. PPV and NPV move a lot, because they depend on how common the
condition is: at 2% prevalence most positive results are false positives. A model's PPV from one
hospital cannot be carried over to a setting with a different case mix.

### Step 3

Plot the ROC curve with sklearn.metrics.RocCurveDisplay and mark your three thresholds on it.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay

fig, ax = plt.subplots(figsize=(4.5, 4.5))
RocCurveDisplay.from_predictions(y_true, y_prob, ax=ax, name="Simulated model")
for t in THRESHOLDS:
    m = metrics_at(y_true, y_prob, t)
    ax.scatter(1 - m["specificity"], m["sensitivity"], zorder=3)
    ax.annotate(f"threshold {t}", (1 - m["specificity"], m["sensitivity"]),
                textcoords="offset points", xytext=(8, -12))
ax.plot([0, 1], [0, 1], linestyle=":", color="grey")
plt.show()

### Step 4

Draw a calibration plot. Is the model over- or under-estimating risk? Recalibrate it with logistic (Platt) scaling and compare Brier scores.

In [ ]:
from sklearn.calibration import CalibrationDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Fit the recalibration on one half and judge it on the other, so the comparison is fair.
logit_score = np.log(y_prob / (1 - y_prob)).reshape(-1, 1)
idx_fit, idx_test = train_test_split(np.arange(len(y_true)), test_size=0.5, random_state=0, stratify=y_true)
platt = LogisticRegression().fit(logit_score[idx_fit], y_true[idx_fit])
recalibrated = platt.predict_proba(logit_score[idx_test])[:, 1]

before = brier_score_loss(y_true[idx_test], y_prob[idx_test])
after = brier_score_loss(y_true[idx_test], recalibrated)
print(f"Mean predicted risk {y_prob[idx_test].mean():.2f} vs observed rate {y_true[idx_test].mean():.2f}")
print(f"Brier score before {before:.3f}, after Platt scaling {after:.3f}")

fig, ax = plt.subplots(figsize=(4.5, 4.5))
CalibrationDisplay.from_predictions(y_true[idx_test], y_prob[idx_test], n_bins=10, strategy="quantile", ax=ax, name="Original")
CalibrationDisplay.from_predictions(y_true[idx_test], recalibrated, n_bins=10, strategy="quantile", ax=ax, name="Platt-scaled")
plt.show()

The original curve lies below the diagonal: the model **over-estimates** risk (it predicts far more events
than happen). Platt scaling leaves the ranking, and so the AUROC, unchanged, but brings the predicted
probabilities in line with observed rates, which lowers the Brier score.

### Step 5

Write two sentences you would put in a report for clinicians describing this model's performance.

Example wording (numbers from the runs above, at 10% prevalence and threshold 0.5):

"In simulated data with 10% prevalence, the model identified about nine in ten patients with the condition
(sensitivity 0.92) and correctly ruled out about 85% of those without it, but only about four in ten
positive results were true positives (PPV 0.41), and this share would be lower where the condition is
rarer. Its original risk estimates were too high and needed recalibration before they could be read as
probabilities; it has not been validated on real patients."

---

Lesson: [https://ramkumarmd.com/medical-ai/foundations/diagnostic-accuracy-metrics](https://ramkumarmd.com/medical-ai/foundations/diagnostic-accuracy-metrics). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.